# E6a: download the MCT-LTDiag HCC and ICC cases into Drive

**Revision plan (approved 17 Oct 2026), data step for experiment E6.** MCT-LTDiag (Peking Union Medical College
Hospital) is a public multiphase-CT liver tumour dataset: Harvard Dataverse doi:10.7910/DVN/S3RW15, **version 3.1, CC0**.
It has five classes. Only **HCC (103 patients)** and **ICC (100 patients)** are downloaded, because these are the two
classes it shares with our label set (it has no combined HCC-ICC class). That is **203 tar files, about 67.6 GB**.

**How the class of each file is known.** The dataset's patient table (`meta_info_patient`) has a `type` column. The
notebook selects files by that column and also checks that it agrees one-to-one with the letter in the file name
(a = BCLM, b = HCC, c = HH, d = ICC, e = CRLM) and that the counts are exactly 103 and 100. Otherwise it stops.

**Checks on every file.**
1. After download: MD5 and size against the checksums published by Dataverse.
2. After upload: Drive's own MD5 of the stored file against the same checksum.
A file that fails is downloaded again (it is never kept). The tar files are **not unpacked** here; the notebook only
reads the list of file names inside each one, so that the E6 preprocessing can be written from facts.

**Runtime:** a **CPU runtime is enough** (no GPU). It can run **at the same time as E5** in a second Colab tab.
Keep this tab open. Time depends on network speed (roughly 30–120 minutes).
**Resume:** if the runtime stops, just run all cells again. Files already in Drive with the correct checksum are skipped.

**Version 2 (17 Oct 2026).** Version 1 stopped at Cell 2 because Harvard Dataverse refuses requests that carry Python's
default user agent (403 Forbidden). Version 2 sends a plain self-identifying user agent on every Dataverse request.
It also reads ages written as `062Y` (62 years), the format used for the HCC and ICC patients. Nothing else changed.
The folders made by version 1's Cell 1 are found and reused.

**Safety:** nothing existing is changed. Everything goes to a new folder
`liver_project/external_validation/MCT-LTDiag/` (subfolders `metadata/`, `tar_HCC/`, `tar_ICC/`, `logs/`).

In [ ]:
# Cell 1 — Setup: sign-in, account and space checks, destination folders
import os, sys, io, re, json, time, gzip, random, shutil, hashlib, tarfile, threading, datetime
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()            # sign in with the account that owns the project folder
    TMP = Path('/content/mct_tmp')
else:
    TMP = Path(os.environ['MCT_TEST_TMP'])
TMP.mkdir(parents=True, exist_ok=True)

import requests
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload

DOI = 'doi:10.7910/DVN/S3RW15'
DATASET_VERSION = '3.1'
DV = 'https://dataverse.harvard.edu'
HEADERS = {'User-Agent': 'liver-revision-download/1.0 (research; Google Colab)'}   # Dataverse refuses the default python-requests agent (403)
CLASSES = ['HCC', 'ICC']                      # the MCT-LTDiag classes that exist in our label set
EXPECTED = {'HCC': 103, 'ICC': 100}           # counts published with the dataset
N_WORKERS = 4                                 # files handled at the same time
CHUNK = 64 * 1024 * 1024
TOP_FOLDER_ID = os.environ['LIVER_DRIVE_PROJECT_FOLDER_ID']        # Drive folder of the project (see docs/revision_notebooks.md)
EXTERNAL_FOLDER_ID = os.environ['LIVER_DRIVE_EXTERNAL_FOLDER_ID']   # .../liver_project/external_validation (holds HCC-TACE-Seg)
TOP_NAME = 'Multimodal Classification of Primary Liver Tumors on Multiphase CT'
FOLDER = 'application/vnd.google-apps.folder'
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')


class Stop(RuntimeError):
    """Raised when a check fails. Nothing has been changed when this is raised before Cell 4."""


_local = threading.local()
def svc():
    """One Drive client per thread (the HTTP layer underneath is not thread-safe)."""
    if not hasattr(_local, 'svc'):
        _local.svc = build('drive', 'v3', cache_discovery=False)
    return _local.svc


def retryable(e):
    status = getattr(getattr(e, 'resp', None), 'status', 0)
    return status in (429, 500, 502, 503, 504) or (status == 403 and 'rate' in str(e).lower())


def call(request, tries=8):
    """Execute a Drive API request, retrying on rate limits and transient errors."""
    for attempt in range(tries):
        try:
            return request.execute()
        except HttpError as e:
            if not retryable(e) or attempt == tries - 1:
                raise
            time.sleep(min(64, 2 ** attempt) + random.random())


def children(folder_id):
    out, token = [], None
    while True:
        res = call(svc().files().list(q=f"'{folder_id}' in parents and trashed = false", pageSize=1000, pageToken=token,
                                      fields='nextPageToken, files(id, name, mimeType, size, md5Checksum)'))
        out += res.get('files', [])
        token = res.get('nextPageToken')
        if not token:
            return out


def ensure_folder(name, parent_id):
    hits = [f for f in children(parent_id) if f['name'] == name and f['mimeType'] == FOLDER]
    if len(hits) > 1:
        raise Stop(f'There are {len(hits)} folders named {name!r} in the same place; send this output for review.')
    if hits:
        return hits[0]['id']
    return call(svc().files().create(body={'name': name, 'mimeType': FOLDER, 'parents': [parent_id]}, fields='id'))['id']


def md5_of(path):
    h = hashlib.md5()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(CHUNK), b''):
            h.update(block)
    return h.hexdigest()


def upload_small(local, name, parent_id, mime):
    """Upload a small file unless an identical one is already there. Never overwrites: a different file with the same
    name is uploaded under a new name with the run time appended."""
    existing = [f for f in children(parent_id) if f['name'] == name]
    digest = md5_of(local)
    if any(f.get('md5Checksum') == digest for f in existing):
        return 'already there'
    if existing:
        stem, dot, ext = name.rpartition('.')
        name = f'{stem}_{RUN_ID}.{ext}' if dot else f'{name}_{RUN_ID}'
    media = MediaFileUpload(str(local), mimetype=mime, resumable=False)
    call(svc().files().create(body={'name': name, 'parents': [parent_id]}, media_body=media, fields='id'))
    return f'uploaded as {name}'


def upsert_log(local, name, parent_id, mime):
    """Create or refresh one of this notebook's own log files in logs/."""
    existing = [f for f in children(parent_id) if f['name'] == name]
    media = MediaFileUpload(str(local), mimetype=mime, resumable=False)
    if existing:
        call(svc().files().update(fileId=existing[0]['id'], media_body=media, fields='id'))
    else:
        call(svc().files().create(body={'name': name, 'parents': [parent_id]}, media_body=media, fields='id'))


# ---- account, destination and space checks (read-only) ----
try:
    me = call(svc().about().get(fields='user(emailAddress), storageQuota'))
    top = call(svc().files().get(fileId=TOP_FOLDER_ID, fields='name, trashed'))
    ext = call(svc().files().get(fileId=EXTERNAL_FOLDER_ID, fields='name, parents, trashed'))
    lp = call(svc().files().get(fileId=ext['parents'][0], fields='name, parents, trashed'))
except HttpError as e:
    raise Stop(f'The project folders are not visible to this account ({e}). Sign in with the account that owns the project folder.')
if (top['name'] != TOP_NAME or top.get('trashed') or ext['name'] != 'external_validation' or ext.get('trashed')
        or lp['name'] != 'liver_project' or lp.get('trashed') or lp['parents'][0] != TOP_FOLDER_ID):
    raise Stop('The destination folders are not where expected; send this output for review.')
print('Signed in as:', me['user']['emailAddress'])

quota = me.get('storageQuota', {})
need_gb = 75
if 'limit' in quota:
    free_gb = (int(quota['limit']) - int(quota['usage'])) / 1e9
    print(f'Drive free space: {free_gb:,.0f} GB (need about {need_gb} GB)')
    if free_gb < need_gb:
        raise Stop('Not enough Drive space for the download.')
else:
    print('Drive free space: no limit reported')
local_free_gb = shutil.disk_usage(TMP).free / 1e9
print(f'Local disk free: {local_free_gb:,.0f} GB')
if local_free_gb < 10:
    raise Stop('Less than 10 GB free on the Colab disk.')

# ---- destination folders (created only if missing) ----
MCT = ensure_folder('MCT-LTDiag', EXTERNAL_FOLDER_ID)
META = ensure_folder('metadata', MCT)
TAR = {c: ensure_folder(f'tar_{c}', MCT) for c in CLASSES}
LOGS = ensure_folder('logs', MCT)
readme = TMP / '00_README_MCT-LTDiag.txt'
readme.write_text(
    'MCT-LTDiag (Peking Union Medical College Hospital), public multiphase CT liver tumour dataset.\n'
    f'Source: Harvard Dataverse {DOI}, version {DATASET_VERSION}, licence CC0 1.0. Downloaded by notebook\n'
    'notebooks/3_revision_2026-10/E6a_download_MCT-LTDiag_HCC_ICC_v2.ipynb for the JIMH revision (external test E6).\n'
    'Only the HCC (103) and ICC (100) patients are kept: tar_HCC/ and tar_ICC/ hold the original tar files, unchanged.\n'
    'Every file was checked against the Dataverse MD5 after download and again after upload (Drive MD5).\n'
    'metadata/ holds the dataset tables as published; logs/ holds the download log, the list of files inside each tar\n'
    'and the final verification. Do not edit or rename files here; analysis outputs go to liver_project/revision_2026-10.\n'
    'Cite the dataset and its paper (Hoyant-Su et al., Multi-phase liver tumour benchmark) when used.\n', encoding='utf-8')
upsert_log(readme, '00_README_MCT-LTDiag.txt', MCT, 'text/plain')     # this notebook's own README, refreshed
print('Writing only inside: liver_project/external_validation/MCT-LTDiag/  (metadata/, tar_HCC/, tar_ICC/, logs/)')

In [ ]:
# Cell 2 — File list and checksums of dataset version 3.1 (read from Dataverse)
r = requests.get(f'{DV}/api/datasets/:persistentId/versions/{DATASET_VERSION}', params={'persistentId': DOI},
                 headers=HEADERS, timeout=120)
if r.status_code != 200:
    raise Stop(f'Dataverse answered {r.status_code} for the file list: {r.text[:300]!r}. Send this output.')
VERSION_JSON = r.json()
ver = VERSION_JSON['data']
if f"{ver['versionNumber']}.{ver['versionMinorNumber']}" != DATASET_VERSION or ver.get('versionState') != 'RELEASED':
    raise Stop(f"Unexpected dataset version {ver.get('versionNumber')}.{ver.get('versionMinorNumber')}")
LICENCE = (ver.get('license') or {}).get('name', 'not stated')

rows = []
for item in ver['files']:
    d = item['dataFile']
    checksum = d.get('checksum') or {'type': 'MD5', 'value': d.get('md5')}
    rows.append({'name': d['filename'], 'dataverse_id': d['id'], 'size': int(d['filesize']), 'md5': checksum['value'],
                 'checksum_type': checksum['type'], 'restricted': bool(item.get('restricted', False)),
                 'original_name': d.get('originalFileName')})
files = pd.DataFrame(rows)
tars = files[files['name'].str.endswith('.tar')].copy()
tars['patient_id'] = tars['name'].str[:-4]
tars['letter'] = tars['patient_id'].str.extract(r'^\d{6}([a-z])\d+$')[0]

problems = []
if len(files) != 520 or len(tars) != 517:
    problems.append(f'{len(files)} files / {len(tars)} tar files (expected 520 / 517)')
if (files['checksum_type'] != 'MD5').any() or files['md5'].isna().any():
    problems.append('some files have no MD5 checksum')
if files['restricted'].any():
    problems.append('some files are restricted')
if tars['letter'].isna().any() or tars['patient_id'].duplicated().any():
    problems.append('unexpected tar file names')
if problems:
    raise Stop('Dataset file list check failed: ' + '; '.join(problems))
print(f'Dataset {DOI} v{DATASET_VERSION} ({LICENCE}): {len(files)} files, {len(tars)} tar files, '
      f'{tars["size"].sum() / 1e9:.1f} GB in total')
print(tars.groupby('letter')['size'].agg(files='count', GB=lambda s: round(s.sum() / 1e9, 2)).to_string())

In [ ]:
# Cell 3 — Dataset tables: checksum, class of every patient, selection of HCC and ICC
def fetch_bytes(file_id, original=False):
    last = None
    for attempt in range(1, 6):
        try:
            resp = requests.get(f'{DV}/api/access/datafile/{file_id}', params={'format': 'original'} if original else None,
                                headers=HEADERS, timeout=(30, 120))
            resp.raise_for_status()
            return resp.content
        except requests.RequestException as e:
            last = e
            time.sleep(5 * attempt)
    raise Stop(f'Could not download Dataverse file {file_id}: {last!r}')


# Dataverse converted the authors' CSV tables to .tab; the original CSV is also served. Both are kept.
TABLES, META_MD5 = {}, {}
for tab_name in ['meta_info_patient.tab', 'meta_info_tumor.tab']:
    row = files[files['name'] == tab_name].iloc[0]
    variants = {'tab': fetch_bytes(row['dataverse_id']), 'original': fetch_bytes(row['dataverse_id'], original=True)}
    matched = [k for k, v in variants.items() if hashlib.md5(v).hexdigest() == row['md5']]
    META_MD5[tab_name] = matched or ['NONE']
    use = 'original' if 'original' in matched or not matched else 'tab'
    raw = variants[use]
    table = (pd.read_csv(io.BytesIO(raw), encoding='utf-8-sig', dtype=str) if use == 'original'
             else pd.read_csv(io.BytesIO(raw), sep='\t', dtype=str))
    table.columns = [c.strip() for c in table.columns]
    TABLES[tab_name] = table
    for kind, blob in variants.items():
        out_name = (row['original_name'] or tab_name.replace('.tab', '.csv')) if kind == 'original' else tab_name
        (TMP / out_name).write_bytes(blob)
    print(f'{tab_name}: Dataverse MD5 matches the {" and ".join(matched) if matched else "NO"} version; '
          f'{len(table)} rows; columns: {list(table.columns)}')
    if not matched:
        print('  WARNING: checksum not matched; the content checks below still decide whether to continue.')

pat = TABLES['meta_info_patient.tab']
if not {'ID', 'type'} <= set(pat.columns):
    raise Stop('The patient table has no ID/type columns.')
pat['ID'] = pat['ID'].str.strip()
pat['type'] = pat['type'].str.strip()
if len(pat) != 517 or pat['ID'].duplicated().any():
    raise Stop(f'Patient table: {len(pat)} rows, {pat["ID"].duplicated().sum()} duplicated IDs (expected 517 unique).')
missing_tar, missing_row = sorted(set(pat['ID']) - set(tars['patient_id'])), sorted(set(tars['patient_id']) - set(pat['ID']))
if missing_tar or missing_row:
    raise Stop(f'IDs without a tar file: {missing_tar[:10]}; tar files without a table row: {missing_row[:10]}')

merged = tars.merge(pat, left_on='patient_id', right_on='ID', how='inner')
cross = pd.crosstab(merged['letter'], merged['type'])
print('\nFile-name letter x type column:\n' + cross.to_string())
if not ((cross > 0).sum(axis=1) == 1).all() or not ((cross > 0).sum(axis=0) == 1).all():
    raise Stop('The file-name letter and the type column do not agree one-to-one.')
LETTER_OF = {t: cross.index[cross[t] > 0][0] for t in cross.columns}
counts = merged['type'].value_counts()
for c in CLASSES:
    if counts.get(c, 0) != EXPECTED[c]:
        raise Stop(f'{c}: {counts.get(c, 0)} patients in the table, expected {EXPECTED[c]}.')


def natural(pid):
    m = re.match(r'^(\d{6})([a-z])(\d+)$', pid)
    return (m.group(1), m.group(2), int(m.group(3)))


SEL = merged[merged['type'].isin(CLASSES)].copy()
SEL['_key'] = SEL['patient_id'].map(natural)
SEL = SEL.sort_values(['type', '_key']).drop(columns=['_key', 'ID']).reset_index(drop=True)
print(f'\nSelected: ' + ', '.join(f'{c} {int((SEL["type"] == c).sum())} (letter {LETTER_OF[c]})' for c in CLASSES)
      + f'; {len(SEL)} tar files, {SEL["size"].sum() / 1e9:.1f} GB')


def col(df, *words):
    for c in df.columns:
        if all(w in c.lower() for w in words):
            return c
    return None


CLIN = {'sex': col(pat, 'sex'), 'age': col(pat, 'age'), 'cirrhosis': col(pat, 'cirrhosis'),
        'viral_hepatitis': col(pat, 'hepatitis'), 'chemotherapy': col(pat, 'chemotherapy')}
clin_summary = {}
for c in CLASSES:
    part = SEL[SEL['type'] == c]
    age = (pd.to_numeric(part[CLIN['age']].str.strip().str.replace(r'^0*(\d+)\s*[Yy]$', r'\1', regex=True),   # '062Y' = 62 years
                         errors='coerce') if CLIN['age'] else pd.Series(dtype=float))
    clin_summary[c] = {'n': len(part), 'age_median': float(age.median()) if age.notna().any() else None,
                       'age_q1_q3': [float(age.quantile(.25)), float(age.quantile(.75))] if age.notna().any() else None,
                       'age_missing': int(age.isna().sum())}
    for key in ['sex', 'cirrhosis', 'viral_hepatitis', 'chemotherapy']:
        if CLIN[key]:
            clin_summary[c][key] = part[CLIN[key]].fillna('missing').value_counts().to_dict()
print('Clinical columns found:', CLIN)
print(json.dumps(clin_summary, indent=1))

tum = TABLES['meta_info_tumor.tab']
if 'ID' in tum.columns:
    tum_sel = tum[tum['ID'].str.strip().isin(SEL['patient_id'])]
    print(f'Tumour table: {len(tum)} rows; {len(tum_sel)} rows for the selected patients '
          f'({tum_sel["ID"].nunique()} unique IDs)')

# ---- keep the tables as published, plus the selection, in metadata/ ----
(TMP / 'dataset_version_3.1_metadata.json').write_text(json.dumps(VERSION_JSON, indent=1), encoding='utf-8')
SEL.to_csv(TMP / 'selected_HCC_ICC_patients.csv', index=False)
nb_row = files[files['name'] == 'MCT_LTDiag_download.ipynb']
if len(nb_row):
    (TMP / 'MCT_LTDiag_download_AUTHORS.ipynb').write_bytes(fetch_bytes(nb_row.iloc[0]['dataverse_id']))  # kept as reference, not run
for name, mime in [('meta_info_patient.csv', 'text/csv'), ('meta_info_patient.tab', 'text/tab-separated-values'),
                   ('meta_info_tumor.csv', 'text/csv'), ('meta_info_tumor.tab', 'text/tab-separated-values'),
                   ('dataset_version_3.1_metadata.json', 'application/json'),
                   ('selected_HCC_ICC_patients.csv', 'text/csv'),
                   ('MCT_LTDiag_download_AUTHORS.ipynb', 'application/x-ipynb+json')]:
    if (TMP / name).exists():
        print(f'metadata/{name}: {upload_small(TMP / name, name, META, mime)}')

In [ ]:
# Cell 4 — Download, check, upload (resumable). Re-run this cell if anything failed.
ACCESS = DV + '/api/access/datafile/{}'
LOG_COLS = ['name', 'patient_id', 'type', 'dataverse_id', 'bytes', 'md5_expected', 'drive_file_id', 'drive_md5',
            'download_attempts', 'download_s', 'upload_s', 'n_members', 'n_member_files', 'listing_error', 'utc']
MEMBER_COLS = ['tar', 'member', 'size', 'kind']


def download(file_id, dest, md5, size, tries=6):
    part = dest.with_name(dest.name + '.part')
    last = ''
    for attempt in range(1, tries + 1):
        try:
            h, n = hashlib.md5(), 0
            with requests.get(ACCESS.format(file_id), stream=True, headers=HEADERS, timeout=(30, 300)) as resp:
                resp.raise_for_status()
                with open(part, 'wb') as handle:
                    for block in resp.iter_content(8 * 1024 * 1024):
                        handle.write(block)
                        h.update(block)
                        n += len(block)
            if n == size and h.hexdigest() == md5:
                part.replace(dest)
                return attempt
            last = f'got {n:,} bytes, MD5 {h.hexdigest()}'
        except Exception as e:
            last = repr(e)[:200]
        part.unlink(missing_ok=True)
        time.sleep(min(120, 5 * 2 ** attempt) + random.random())
    raise RuntimeError(f'download failed {tries} times ({last})')


def list_members(path):
    """Names inside the tar, read without unpacking anything."""
    out = []
    try:
        with tarfile.open(path, 'r:*') as tf:
            for m in tf:
                out.append((m.name, int(m.size), 'file' if m.isfile() else 'dir' if m.isdir() else 'other'))
        return out, ''
    except (tarfile.TarError, OSError, EOFError) as e:
        return out, repr(e)[:200]


def upload_tar(path, name, parent_id):
    media = MediaFileUpload(str(path), mimetype='application/x-tar', resumable=True, chunksize=CHUNK)
    request = svc().files().create(body={'name': name, 'parents': [parent_id]}, media_body=media,
                                   fields='id, name, size, md5Checksum')
    response, failures = None, 0
    while response is None:
        try:
            _, response = request.next_chunk(num_retries=3)
        except HttpError as e:
            if not retryable(e) or failures >= 8:
                raise
            failures += 1
            time.sleep(min(64, 2 ** failures) + random.random())
        except Exception:                 # dropped connection: the resumable upload continues where it stopped
            if failures >= 8:
                raise
            failures += 1
            time.sleep(min(64, 2 ** failures) + random.random())
    return response


QUARANTINE = None
def quarantine(file_id, parent_id):
    """Move a wrong or second copy out of tar_*/ into logs/quarantine/ (moved, not deleted)."""
    global QUARANTINE
    with LOCK:
        if QUARANTINE is None:
            QUARANTINE = ensure_folder('quarantine', LOGS)
    call(svc().files().update(fileId=file_id, addParents=QUARANTINE, removeParents=parent_id, fields='id'))


def process(r):
    local = TMP / r.name
    try:
        t0 = time.time()
        attempts = download(r.dataverse_id, local, r.md5, r.size)
        t1 = time.time()
        members, listing_error = list_members(local)
        up = upload_tar(local, r.name, TAR[r.type])
        t2 = time.time()
    finally:
        local.unlink(missing_ok=True)
        local.with_name(local.name + '.part').unlink(missing_ok=True)
    if up.get('md5Checksum') != r.md5 or int(up.get('size', -1)) != r.size:
        quarantine(up['id'], TAR[r.type])
        raise RuntimeError(f'Drive checksum after upload was {up.get("md5Checksum")}, expected {r.md5}')
    record = {'name': r.name, 'patient_id': r.patient_id, 'type': r.type, 'dataverse_id': r.dataverse_id,
              'bytes': r.size, 'md5_expected': r.md5, 'drive_file_id': up['id'], 'drive_md5': up['md5Checksum'],
              'download_attempts': attempts, 'download_s': round(t1 - t0, 1), 'upload_s': round(t2 - t1, 1),
              'n_members': len(members), 'n_member_files': sum(m[2] == 'file' for m in members),
              'listing_error': listing_error, 'utc': datetime.datetime.now(datetime.timezone.utc).isoformat()}
    return record, [(r.name,) + m for m in members]


def process_with_retries(r, tries=3):
    for attempt in range(1, tries + 1):
        try:
            return process(r)
        except Exception:
            if attempt == tries:
                raise
            time.sleep(10 * attempt)


def read_log(name, **kw):
    hit = [f for f in children(LOGS) if f['name'] == name]
    if not hit:
        return None
    return pd.read_csv(io.BytesIO(call(svc().files().get_media(fileId=hit[0]['id']))), **kw)


def push_logs():
    log_df = pd.DataFrame(LOG_ROWS, columns=LOG_COLS).drop_duplicates('name', keep='last')
    log_df.to_csv(TMP / 'download_log.csv', index=False)
    mem_df = pd.DataFrame(MEMBER_ROWS, columns=MEMBER_COLS).drop_duplicates(['tar', 'member'], keep='last')
    mem_df.to_csv(TMP / 'tar_member_index.csv.gz', index=False, compression='gzip')
    upsert_log(TMP / 'download_log.csv', 'download_log.csv', LOGS, 'text/csv')
    upsert_log(TMP / 'tar_member_index.csv.gz', 'tar_member_index.csv.gz', LOGS, 'application/gzip')


LOCK = threading.Lock()
old_log = read_log('download_log.csv', dtype={'patient_id': str})
old_mem = read_log('tar_member_index.csv.gz', compression='gzip')
LOG_ROWS = [] if old_log is None else old_log.to_dict('records')
MEMBER_ROWS = [] if old_mem is None else list(old_mem[MEMBER_COLS].itertuples(index=False, name=None))

# ---- what is already in Drive ----
stored = {}
for c in CLASSES:
    for f in children(TAR[c]):
        stored.setdefault((c, f['name']), []).append(f)
to_download, skipped = [], 0
for r in SEL.itertuples():
    hits = stored.pop((r.type, r.name), [])
    good = [f for f in hits if f.get('md5Checksum') == r.md5 and int(f.get('size', -1)) == r.size]
    keep = good[0] if good else None
    for f in hits:
        if f is not keep:
            why = 'second copy' if keep is not None and f.get('md5Checksum') == r.md5 else 'wrong checksum'
            quarantine(f['id'], TAR[r.type])
            print(f'moved to logs/quarantine/: {r.name} ({why})')
    if good:
        skipped += 1
    else:
        to_download.append(r)
for (c, name), hits in stored.items():
    print(f'NOTE: tar_{c}/{name} is not one of the selected files; left untouched.')
total = sum(r.size for r in to_download)
print(f'Already in Drive and verified: {skipped}. To download now: {len(to_download)} files, {total / 1e9:.1f} GB '
      f'with {N_WORKERS} at a time.\n')

FAILED, done_bytes, started = [], 0, time.time()
if to_download:
    with ThreadPoolExecutor(N_WORKERS) as pool:
        futures = {pool.submit(process_with_retries, r): r for r in to_download}
        for k, fut in enumerate(as_completed(futures), start=1):
            r = futures[fut]
            try:
                record, members = fut.result()
                with LOCK:
                    LOG_ROWS.append(record)
                    MEMBER_ROWS.extend(members)
                done_bytes += r.size
                rate = done_bytes / max(1.0, time.time() - started)
                left = (total - done_bytes) / rate / 60 if rate else float('nan')
                print(f'[{k}/{len(to_download)}] {r.type} {r.name} {r.size / 1e6:,.0f} MB ok '
                      f'(down {record["download_s"]:.0f}s, up {record["upload_s"]:.0f}s) | {done_bytes / 1e9:.1f}/'
                      f'{total / 1e9:.1f} GB | {rate / 1e6:.0f} MB/s | about {left:.0f} min left', flush=True)
            except Exception as e:
                FAILED.append({'name': r.name, 'error': repr(e)[:300]})
                print(f'[{k}/{len(to_download)}] FAILED {r.name}: {e!r}'[:400], flush=True)
            if k % 10 == 0:
                push_logs()
push_logs()
print(f'\nFinished in {(time.time() - started) / 60:.1f} min. Failed: {len(FAILED)}'
      + ('  -> run this cell again to retry them.' if FAILED else ''))

In [ ]:
# Cell 5 — Final check of everything in Drive, summary of what is inside the tar files
present = {}
for c in CLASSES:
    for f in children(TAR[c]):
        present.setdefault((c, f['name']), []).append(f)
ver_rows = []
for r in SEL.itertuples():
    hits = present.pop((r.type, r.name), [])
    ok = len(hits) == 1 and hits[0].get('md5Checksum') == r.md5 and int(hits[0].get('size', -1)) == r.size
    ver_rows.append({'name': r.name, 'type': r.type, 'copies_in_drive': len(hits), 'md5_expected': r.md5,
                     'md5_drive': hits[0].get('md5Checksum') if hits else None, 'bytes': r.size, 'ok': ok})
VER = pd.DataFrame(ver_rows)
EXTRA = sorted(f'tar_{c}/{n}' for c, n in present)    # files this notebook did not put there: reported, left alone
PASS = bool(VER['ok'].all())
VER.to_csv(TMP / f'verification_{RUN_ID}.csv', index=False)
upload_small(TMP / f'verification_{RUN_ID}.csv', f'verification_{RUN_ID}.csv', LOGS, 'text/csv')

mem = pd.DataFrame(MEMBER_ROWS, columns=MEMBER_COLS).drop_duplicates(['tar', 'member'], keep='last')
mem = mem[mem['tar'].isin(VER.loc[VER['ok'], 'name'])]


def pattern(row):
    pid = row['tar'][:-4]
    s = row['member'].replace(pid, '<ID>')
    s = re.sub(r'[\d.]{6,}', '#', s)              # DICOM UIDs and long numbers
    return re.sub(r'\d{3,}', '#', s)              # slice numbers


files_only = mem[mem['kind'] == 'file'].copy()
STRUCTURE = {}
if len(files_only):
    files_only['pattern'] = files_only.apply(pattern, axis=1)
    files_only['ext'] = files_only['member'].str.lower().str.extract(r'(\.nii\.gz|\.[a-z0-9]{1,5})$')[0].fillna('(none)')
    per_tar = files_only.groupby('tar').size()
    STRUCTURE = {
        'tars_listed': int(mem['tar'].nunique()),
        'files_per_tar_min_median_max': [int(per_tar.min()), float(per_tar.median()), int(per_tar.max())],
        'extensions': files_only['ext'].value_counts().to_dict(),
        'top_patterns': (files_only.groupby('pattern')['tar'].agg(files='size', tars='nunique')
                         .sort_values('tars', ascending=False).head(25).reset_index().to_dict('records')),
    }
summary = {'run': RUN_ID, 'dataset': f'{DOI} v{DATASET_VERSION}', 'licence': LICENCE, 'pass': PASS,
           'verified': {c: int(VER[(VER['type'] == c) & VER['ok']].shape[0]) for c in CLASSES},
           'expected': EXPECTED, 'gigabytes': round(float(VER.loc[VER['ok'], 'bytes'].sum()) / 1e9, 2),
           'not_ok': VER.loc[~VER['ok'], 'name'].tolist(), 'extra_files': EXTRA, 'metadata_md5_matched': META_MD5,
           'letter_of_class': LETTER_OF, 'clinical_columns': CLIN, 'clinical_summary': clin_summary,
           'listing_errors': int((pd.DataFrame(LOG_ROWS, columns=LOG_COLS)['listing_error'].fillna('') != '').sum()),
           'structure': STRUCTURE}
(TMP / f'summary_{RUN_ID}.json').write_text(json.dumps(summary, indent=1, default=str), encoding='utf-8')
upload_small(TMP / f'summary_{RUN_ID}.json', f'summary_{RUN_ID}.json', LOGS, 'application/json')

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'E6a MCT-LTDiag download | run {RUN_ID} | {"PASS" if PASS else "NOT COMPLETE"} | verified HCC '
      f'{summary["verified"]["HCC"]}/{EXPECTED["HCC"]}, ICC {summary["verified"]["ICC"]}/{EXPECTED["ICC"]} | '
      f'{summary["gigabytes"]} GB')
print('Metadata MD5 matched:', META_MD5, '| letters:', LETTER_OF, '| licence:', LICENCE)
if summary['not_ok']:
    print('Not OK (run Cell 4 again):', summary['not_ok'][:20])
if EXTRA:
    print('NOTE, other files in tar_*/ (not touched):', EXTRA[:20])
print('Clinical columns:', CLIN)
print('Clinical summary:', json.dumps(clin_summary))
print('Tar listing errors:', summary['listing_errors'])
if STRUCTURE:
    print('Files per tar (min/median/max):', STRUCTURE['files_per_tar_min_median_max'])
    print('Extensions:', STRUCTURE['extensions'])
    print('Most common paths inside the tars (<ID> = patient ID, # = number):')
    for p in STRUCTURE['top_patterns']:
        print(f"  {p['tars']:>4} tars  {p['files']:>6} files  {p['pattern']}")
    first = VER.loc[VER['ok'], 'name'].iloc[0]
    print(f'Example, all entries of {first}:')
    for row in mem[mem['tar'] == first].head(40).itertuples():
        print(f'  {row.kind:5} {row.size:>12,}  {row.member}')
print('==============================================================')